# ML Jobs: Running Training with `submit_directory`

**Session 3 — Deep Dive: Distributed Training & Hyperparameter Optimization**  
**Notebook 1 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **ML Jobs** | Submit local Python scripts to run on Snowflake compute pools |
| **`submit_directory`** | Package a directory of scripts and submit as one job |
| **Environment Variables** | Pass configuration to remote jobs via env vars |
| **Job Lifecycle** | Submit → monitor → retrieve logs → inspect results |
| **Snowpark Session** | `Session.builder.getOrCreate()` inside SPCS containers |

---

## Why ML Jobs?

Training models locally works for prototyping, but production ML needs:

- **Scalable compute** — GPUs and high-memory instances on demand
- **Data proximity** — compute runs next to the data, no egress costs
- **Reproducibility** — consistent runtime environment every time
- **Governance** — training runs inside Snowflake's security perimeter

ML Jobs let you take your existing Python scripts and run them on **Snowpark Container Services (SPCS)**
compute pools with a single API call. No Docker knowledge required for this approach.

### Two Submission Methods

| Method | Use When |
|--------|----------|
| `submit_file()` | Single standalone script, no local imports |
| `submit_directory()` | Multi-file project with shared modules |

In this notebook we use **`submit_directory()`** because real-world training
projects almost always have multiple files.

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import json
import logging
import os
import time

from utils import get_config
from utils import get_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
COMPUTE_POOL = config.compute.compute_pool
STAGE = f"{DB}.{SCHEMA}.{config.stages.job_payloads}"

session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(f"Context: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## 2 | Inspect the Training Script

Before submitting a job, let's look at what we're sending. The `job_payload/` directory
contains a self-contained training project:

```
job_payload/
├── modules/                # Shared classes used by all entrypoints
│   ├── __init__.py
│   ├── preprocess.py       # Preprocessor class (column defs, ColumnTransformer, LabelEncoder)
│   └── train.py            # Trainer class (train, evaluate, register)
├── train_simple.py         # Entrypoint: loads data, trains RF, registers model
├── train_xgb_docker.py     # Entrypoint for Dockerfile demo (notebook 02)
├── train_hpo_tuner.py      # Entrypoint for HPO demo (notebook 04)
├── requirements.txt        # Dependencies (scikit-learn, xgboost)
└── Dockerfile              # Custom image definition (notebook 02)
```

### Key Design Patterns in `train_simple.py`

1. **`Session.builder.getOrCreate()`** — inside SPCS, the session is auto-available
2. **Environment variables** — configuration injected via `env_vars` parameter
3. **Model Registry** — the script registers the trained model directly from the container
4. **Shared modules** — `Preprocessor` and `Trainer` classes in `modules/` are reused across all entrypoints
5. **Standard libraries** — uses sklearn, pandas — no Snowflake-specific training APIs required

## 3 | Submit the ML Job

### How `submit_directory` Works

```
 Local Machine                     Snowflake (SPCS)
 ┌──────────────┐                   ┌───────────────────────┐
 │ job_payload/ │  submit_directory │ Compute Pool          │
 │  train.py    │ ───────────────>  │  ┌─────────────────┐  │
 │  reqs.txt    │   uploads to      │  │ Container       │  │
 │  ...         │   stage, then     │  │ pip install ... │  │
 └──────────────┘   schedules on    │  │  python train.py│  │
                    compute pool    │  └─────────────────┘  │
                                    └───────────────────────┘
```

### Key Parameters

| Parameter | Purpose |
|-----------|---------|
| `dir_path` | Local directory to upload |
| `entrypoint` | Which `.py` file to run |
| `compute_pool` | SPCS pool for execution |
| `stage_name` | Snowflake stage for payload upload |
| `env_vars` | Config passed as environment variables |
| `target_instances` | Number of nodes (1 = single, >1 = distributed) |
| `pip_requirements` | Extra packages (auto-detected from requirements.txt) |

In [ ]:
from datetime import datetime
from snowflake.ml.feature_store import FeatureStore

fs = FeatureStore(session, DB, SCHEMA, default_warehouse=WH)
fv = fs.get_feature_view("PATIENT_FEATURES", "v1")

spine_df = session.table(f"{DB}.{SCHEMA}.RAW_PATIENT_DATA").select(
    "PATIENT_ID", "TIMESTAMP"
)

DATASET_NAME = "PATIENT_TRAINING_DATASET"
DATASET_VERSION = datetime.now().strftime("v_%Y%m%d_%H%M%S")

ds = fs.generate_dataset(
    name=f"{DB}.{SCHEMA}.{DATASET_NAME}",
    spine_df=spine_df,
    features=[fv],
    spine_timestamp_col="TIMESTAMP",
    version=DATASET_VERSION,
)

print(f"Generated dataset: {DATASET_NAME} / {DATASET_VERSION}")
print(f"  Rows: {ds.read.to_pandas().shape[0]}")

In [ ]:
from snowflake.ml.jobs import submit_directory

JOB_DIR = "session3_distributed_training_and_hpo/job_payload"

job = submit_directory(
    dir_path=JOB_DIR,
    entrypoint="train_simple.py",
    compute_pool=COMPUTE_POOL,
    stage_name=STAGE,
    target_instances=1,
    env_vars={
        "DATABASE": DB,
        "SCHEMA": SCHEMA,
        "MODEL_NAME": "PATIENT_RISK_MODEL",
        "N_ESTIMATORS": "150",
        "TRAINING_DATASET_NAME": DATASET_NAME,
        "TRAINING_DATASET_VERSION": DATASET_VERSION,
    },
)

print(f"Job submitted!")
print(f"  Job ID:  {job.id}")
print(f"  Status:  {job.status}")

## 4 | Monitor the Job

ML Jobs are asynchronous. You can:

- **Poll status** — check periodically until done
- **Block with `job.wait()`** — wait for completion
- **Stream logs** — watch training output in real-time
- **List all jobs** — see history of submitted jobs

In [ ]:
print(f"Waiting for job {job.id} ...")
job.wait()
print(f"Final status: {job.status}")

In [ ]:
logs = job.get_logs()
print("=" * 60)
print("JOB LOGS")
print("=" * 60)
print(logs[-3000:] if len(logs) > 3000 else logs)

### List Recent Jobs

You can always retrieve past jobs to inspect logs or results.

In [ ]:
from snowflake.ml.jobs import list_jobs

recent_jobs = list_jobs(limit=5)
display(recent_jobs)

## 5 | Verify the Registered Model

The training script registered the model directly from the SPCS container.
Let's confirm it appeared in the Model Registry.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session, database_name=DB, schema_name=SCHEMA)
model = registry.get_model("PATIENT_RISK_MODEL_SESSION3")

print(f"Model: {model.name}")
print(f"Versions:")
for v in model.versions():
    metrics = v.show_metrics()
    print(
        f"  {v.version_name} — accuracy: {metrics.get('accuracy', 'N/A'):.4f}, "
        f"f1_macro: {metrics.get('f1_macro', 'N/A'):.4f}"
    )

## 6 | Job Management

ML Jobs can be managed programmatically:

In [ ]:
from snowflake.ml.jobs import delete_job, get_job

retrieved_job = get_job(job.id)
print(f"Retrieved job: {retrieved_job.id}")
print(f"Status: {retrieved_job.status}")

# delete_job(retrieved_job)  # Uncomment to clean up

## 7 | Summary

In this notebook we:

1. **Inspected** a self-contained training script designed for ML Jobs
2. **Submitted** the script to SPCS via `submit_directory()`
3. **Monitored** the job status and retrieved logs
4. **Verified** the model was registered in the Snowflake Model Registry
5. **Explored** job management APIs (list, get, delete)

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Zero Docker required** | `submit_directory` handles packaging automatically |
| **Standard Python** | Use sklearn, XGBoost, etc. — no framework lock-in |
| **Automatic session** | `Session.builder.getOrCreate()` works inside SPCS |
| **Env var config** | Pass parameters without modifying scripts |
| **Built-in deps** | `requirements.txt` auto-installed at job start |

---

**Next →** [02_ml_jobs_dockerfile.ipynb](02_ml_jobs_dockerfile.ipynb) — Package training in a custom Docker image